In [ ]:
!pip install git+https://github.com/huggingface/trl.git

  Cloning https://github.com/huggingface/trl.git to /tmp/pip-req-build-dci48ls3
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/trl.git /tmp/pip-req-build-dci48ls3
  Resolved https://github.com/huggingface/trl.git to commit ec1802e3a5c05cf802746c116cc7ac8531ac3f62
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from datasets import load_dataset
from trl.experimental.orpo import ORPOConfig, ORPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
import torch
import os
from datasets import load_dataset
from transformers import AutoTokenizer
from peft import LoraConfig, get_peft_model
from peft import PeftModel, PeftConfig

In [ ]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

login(token="your_hf_token")

dataset = load_dataset("csv", data_files={"train": "./preference_dataset.csv"})
train_dataset = dataset["train"]

train_dataset = train_dataset.shuffle(seed=42).select(range(min(50, len(train_dataset))))

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    "ufert/mistral-ai-tutor",
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained("ufert/mistral-ai-tutor")
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

training_args = ORPOConfig(
    output_dir="orpo",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
)

trainer = ORPOTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=train_dataset
)

trainer.train()


/tmp/ipykernel_4349/3142747527.py:2: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.orpo import ORPOConfig, ORPOTrainer
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrai

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Check 1
Check 2


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1128 > 1024). Running this sequence through the model will result in indexing errors


Check 3


Step,Training Loss
10,117.880444
20,0.000000
30,0.000000


Check 4


In [ ]:
model.save_pretrained("path_to_save_your_model")
tokenizer.save_pretrained("path_to_save_your_model")

In [ ]:
model.push_to_hub("username/model_name")
tokenizer.push_to_hub("username/model_name")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         |  148kB / 13.6MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/ufert/math-ai-tutor-orpo-lora/commit/85d87aa51bdba4c766a0e44d3478fbf34b04a249', commit_message='Upload tokenizer', commit_description='', oid='85d87aa51bdba4c766a0e44d3478fbf34b04a249', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ufert/math-ai-tutor-orpo-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='ufert/math-ai-tutor-orpo-lora'), pr_revision=None, pr_num=None)